In [257]:
%matplotlib auto
import cv2
import numpy as np
from IPython.display import display
from PIL import Image

Using matplotlib backend: inline


In [258]:
CHESS_ROW_SZ = 8
CHESS_COL_SZ = 8
AMT_CHESS_PIECE = 6

# piece index and maximum amount of the piece chess per side
PID_PAWN   = 0; MAX_AMT_PAWN   = 8
PID_ROOK   = 1; MAX_AMT_ROOK   = 2
PID_KNIGHT = 2; MAX_AMT_KNIGHT = 2
PID_BISHOP = 3; MAX_AMT_BISHOP = 2
PID_KING   = 4; MAX_AMT_KING   = 1
PID_QUEEN  = 5; MAX_AMT_QUEEN  = 1
PID_EMPTY  = 6; MAX_AMT_EMPTY  = 64

CID_WHITE = 0
CID_BLACK = 1

CAPTURE_POS = (CHESS_ROW_SZ, CHESS_COL_SZ)
DUMMY_MOVE  = (CAPTURE_POS, CAPTURE_POS)
MAXIMUM_WINDOW_SZ = 60

In [259]:
class BoardData:
    # it contains current status of  the board
    # first two first dimension is board slot and the value is the piece chess type
    

    # add frame to window
    def addFrame(self, frame, cid):
        
        # select frame and window 
        desFrames  = self.whiteFrames   if cid == CID_WHITE else self.blackFrames
        desVotes   = self.curWhiteVotes if cid == CID_WHITE else self.curBlackVotes

        # delete old frame from vote and replace new frame
        oldVoteBoard = self.convertFrameToVoteBoard(desFrames[self.nextIdxOfFrame[cid]])
        desVotes    -= oldVoteBoard
        
        # add to voteWindow
        desFrames[self.nextIdxOfFrame[cid],:,:] = frame
        voteBoard = self.convertFrameToVoteBoard(frame)
        desVotes += voteBoard
        self.nextIdxOfFrame[cid] = (self.nextIdxOfFrame[cid] + 1)% self.WindowSize

    def convertFrameToVoteBoard(self, frame):
        voteBoard  = np.zeros  ((CHESS_ROW_SZ, CHESS_COL_SZ, AMT_CHESS_PIECE + 1), dtype=np.int32)
        rows, cols = np.indices((CHESS_ROW_SZ, CHESS_COL_SZ))
        voteBoard[rows, cols, frame] = 1
        return voteBoard

    def doMajorityVote(self, cid):
        desVotes =  self.curWhiteVotes if cid == CID_WHITE else self.curBlackVotes
        
        piecesResultIdx     = np.argmax(desVotes, axis=2)
        piecesResultVoteVal = np.max   (desVotes, axis=2)

        return piecesResultIdx, piecesResultVoteVal
    
    def isVoteConfident(self, VoteValue):

        amt_confidentEle = int((self.confidentValue/100)*self.WindowSize)
        #print(VoteValue >= amt_confidentEle)
        preRes = np.all(VoteValue >= amt_confidentEle)
        #print("preRes ", preRes)
        return preRes

    def compareDiff(self, frame, cid): 
        
        # desX, desY = (8,8) in case it lose
        # return NOne incase there is no change

        srcState = self.curStateWhite if cid == CID_WHITE else self.curStateBlack
        desState = frame
        
        # difboard
        diffBoard = desState - srcState

        blankToPieces =  np.where(diffBoard < 0)
        pieceToBlanks =  np.where(diffBoard > 0)
        
        btpRes = [(row, col) 
                  for row, col in zip(blankToPieces[0], blankToPieces[1])
        ]
        
        ptbRes = [(row, col) 
                  for row, col in zip(pieceToBlanks[0], pieceToBlanks[1])
        ]


        return btpRes, ptbRes
    
    def createMove(self, blankToPieces, pieceToBlanks):
        # return ((srcX,srcY), (desX, desY))
        #print(blankToPieces, pieceToBlanks)
        b2p_sz = len(blankToPieces)
        p2b_sz = len(pieceToBlanks)

        #### it  only can changed once at a time 
        if (b2p_sz > 1 or p2b_sz > 1):
            return False, None

        #### filter there is unknow occur 
        if (b2p_sz == 1 and p2b_sz == 0):
            return False, None
        
        #### b2p_sz = 0  but p2b_sz = 1  it means it have been captured
        #### b2p_sz = 1  but p2b_sz = 1  it means it just move
        #### b2p_sz = 0  but p2b_sz = 0  it means it is stable
        if ( (b2p_sz == 0) and (p2b_sz == 1)):
            return True, (tuple(pieceToBlanks[0]), CAPTURE_POS)
        if ( (b2p_sz == 1) and (p2b_sz == 1)):
            return True, (tuple(pieceToBlanks[0]), tuple(blankToPieces[0]))
        if ( (b2p_sz == 0) and (p2b_sz == 0)):
            return True, (CAPTURE_POS, CAPTURE_POS)
        
        
        return False, None
    

    def isMoveCaptured(self, move):
        return (move[1] == CAPTURE_POS) and \
               (move[0] != CAPTURE_POS)
    
    def isValidMove(self, whiteMove, blackMove):

        isWhiteCaptured = self.isMoveCaptured(whiteMove)
        isBlackCaptured = self.isMoveCaptured(blackMove)

        isWhiteDummyMove = whiteMove == DUMMY_MOVE
        isBlackDummyMove = blackMove == DUMMY_MOVE

        # detect that if it have been capture 2 side it must be something wrong
        if (isWhiteCaptured and isBlackCaptured):
            return False
        elif isWhiteCaptured:
            return whiteMove[0] == blackMove[1]
        elif isBlackCaptured:
            return blackMove[0] == whiteMove[1]
        else: # it only one can be dummy move
            # both cannot move at the same time
            return isWhiteDummyMove or isBlackDummyMove
    
    def findChessId(self, move, cid):
        # we are so sure that move must have src value
        srcState = self.curStateWhite if cid == CID_WHITE else self.curStateBlack
        srcPos = move[0]

        #print(srcPos)
        if (srcPos == CAPTURE_POS):
            return False, PID_EMPTY
        return True, srcState[srcPos[0], srcPos[1]]

    def updateState(self, move, cid, pieceIdx):
        
        srcState = self.curStateWhite if cid == CID_WHITE else self.curStateBlack
        srcState[move[0][0], move[0][1]] = PID_EMPTY
        if (move[1] != CAPTURE_POS):
            srcState[move[1][0], move[1][1]] = pieceIdx


    def step(self, whiteFrame, blackFrame): 

        # add frame
        self.addFrame(whiteFrame, CID_WHITE)
        self.addFrame(blackFrame, CID_BLACK)

        # white piece ident, white piece vote value
        wFrame, wVoteVal = self.doMajorityVote(CID_WHITE)
        bFrame, bVoteVal = self.doMajorityVote(CID_BLACK)

        #print(wFrame)
        #print(wVoteVal)
        #print("confident vote pass v ", wFrame[3, 3], " amt ", wVoteVal[3, 3])
        if ((not self.isVoteConfident(wVoteVal)) or 
            (not self.isVoteConfident(bVoteVal))
            ):
            return None
        
        #print("pass")
        
        #print("confident vote pass v ", wFrame[3, 3], " amt ", wVoteVal[3, 3])
        


        whiteE2P, whiteP2E = self.compareDiff(wFrame, CID_WHITE)
        blackE2P, blackP2E = self.compareDiff(bFrame, CID_BLACK)

        #print(whiteE2P, whiteP2E)
        #print(blackE2P, blackP2E)

        wMoveValid, wMove = self.createMove(whiteE2P, whiteP2E)
        bMoveValid, bMove = self.createMove(blackE2P, blackP2E)

        if ((not wMoveValid) or (not bMoveValid)):
            return None
        if (not self.isValidMove(wMove, bMove)):
            return None
        #print("wMove is ", wMove, " bMove is ", bMove)
        #print("move check pass")

        if ((wMove == DUMMY_MOVE) and (bMove == DUMMY_MOVE)):
            return None
        
        #### we are now sure that we must have the move

        wMoveChessIdValid, wMoveChessId = self.findChessId(wMove, CID_WHITE)
        bMoveChessIdValid, bMoveChessId = self.findChessId(bMove, CID_BLACK)

        if (wMoveChessIdValid):
            self.updateState(wMove, CID_WHITE, wMoveChessId)
        if (bMoveChessIdValid):
            self.updateState(bMove, CID_BLACK, bMoveChessId)


        return (wMove, wMoveChessId), (bMove, bMoveChessId)
    

    def initBoard(self, frames, cid):
        srcState   = self.curStateWhite if cid == CID_WHITE else self.curStateBlack
        desFrames  = self.whiteFrames   if cid == CID_WHITE else self.blackFrames
        desVotes   = self.curWhiteVotes if cid == CID_WHITE else self.curBlackVotes
        #print(type(srcState))
        # put data to frames/voteboard
        for idx, frame in enumerate(frames):
            desFrames[idx] = frame
            voteBoard = self.convertFrameToVoteBoard(frame)
            #print("vote board ", voteBoard.shape)
            desVotes += voteBoard

        state, voteResult = self.doMajorityVote(cid)
        if not self.isVoteConfident(voteResult):
            print("warning init cid {cid} have not accurate vote")
        if (cid == CID_WHITE):
            self.curStateWhite = state
        else: # black
            self.curStateBlack = state
        

    def __init__(self, whiteFrames, blackFrames, param_windowSize, param_confidentValue):
        self.WindowSize     = param_windowSize
        self.confidentValue = param_confidentValue


        self.curStateBlack = np.zeros((CHESS_ROW_SZ, CHESS_COL_SZ), dtype=np.int8)
        self.curStateWhite = np.zeros((CHESS_ROW_SZ, CHESS_COL_SZ), dtype=np.int8)
        # frame window that used to track the vote from picture frame
        # assume that the window must full at the start
    

        self.curWhiteVotes = np.zeros((CHESS_ROW_SZ, CHESS_COL_SZ, AMT_CHESS_PIECE + 1), dtype=np.int32) # we add dummy size for 1 pieces
        self.curBlackVotes = np.zeros((CHESS_ROW_SZ, CHESS_COL_SZ, AMT_CHESS_PIECE + 1), dtype=np.int32) # we add dummy size for 1 pieces

        ##### first for white and second for black
        self.nextIdxOfFrame = [0,0]
        self.whiteFrames = np.zeros((MAXIMUM_WINDOW_SZ, CHESS_ROW_SZ, CHESS_COL_SZ), dtype=np.int8)
        self.blackFrames = np.zeros((MAXIMUM_WINDOW_SZ, CHESS_ROW_SZ, CHESS_COL_SZ), dtype=np.int8)

        self.initBoard(whiteFrames, CID_WHITE)
        self.initBoard(blackFrames, CID_BLACK)

In [260]:
def duplicateFrame(frame, amt):
    results = np.empty((amt, CHESS_ROW_SZ, CHESS_COL_SZ), dtype=np.int32)
    results[:] = frame
    return results


def createEmptyBoard():
    return np.full((CHESS_ROW_SZ, CHESS_ROW_SZ), PID_EMPTY, dtype=np.int32)

def timeSeriePrint(results):

    if (len(results) == 0):
        print("tp Empty")
    
    lastAnswer = results[0]
    cnt = 1
    for result in results[1:] :
        if result != lastAnswer:
            print(lastAnswer, "  :  ", cnt)
            lastAnswer = result
            cnt = 1
        else:
            cnt = cnt + 1

    print(lastAnswer, "   :   ", cnt)

def printAddr(board):
    print("board.curStateBlack ", id(board.curStateBlack))
    print("board.curStateWhite ", id(board.curStateWhite))
    print("board.curWhiteVotes ", id(board.curWhiteVotes))
    print("board.curBlackVotes ", id(board.curBlackVotes))
    print("board.whiteFrames ", id(board.whiteFrames))
    print("board.blackFrames ", id(board.blackFrames))


In [261]:
def testCase0():
    print("test case 0")
    windowsSize = 60

    initWhiteFrame = createEmptyBoard()
    initWhiteFrame[4,4] = PID_KING
    initBlackFrame = createEmptyBoard()
    initBlackFrame[5,5] = PID_KING

    whiteFrames = duplicateFrame(initWhiteFrame, windowsSize)
    blackFrames = duplicateFrame(initBlackFrame, windowsSize)

    board = BoardData(whiteFrames, blackFrames, windowsSize, 75)
    #printAddr(board)

    # print("-------First State is-------")
    # print("white")
    # print(board.curStateWhite)
    # print("black")
    # print(board.curStateBlack)


    newWhiteFrame = createEmptyBoard()
    newWhiteFrame[3,3] = PID_KING
    newBlackFrame = createEmptyBoard()
    newBlackFrame[5,5] = PID_KING
    
    sumResults = []
    for i in range(46):
        #print(i)
        theResult = board.step(newWhiteFrame, newBlackFrame)
        sumResults.append(theResult)
    #print("move is")
    print("----------result----------")
    timeSeriePrint(sumResults)
    print("--------------------------")
    # print(len(sumResults))
    # print("-------First State is-------")
    # print("white")
    # print(board.curStateWhite)
    # print("black")
    # print(board.curStateBlack)

    



In [262]:
def testCase1():
    print("test case 1")
    windowsSize = 60

    initWhiteFrame = createEmptyBoard()
    initWhiteFrame[4,4] = PID_KING
    initBlackFrame = createEmptyBoard()
    initBlackFrame[5,5] = PID_KING

    whiteFrames = duplicateFrame(initWhiteFrame, windowsSize)
    blackFrames = duplicateFrame(initBlackFrame, windowsSize)

    board = BoardData(whiteFrames, blackFrames, windowsSize, 75)
    #printAddr(board)

    newWhiteFrame = createEmptyBoard()
    newWhiteFrame[5,5] = PID_KING
    newBlackFrame = createEmptyBoard() ### black king is captured
    
    
    sumResults = []
    for i in range(46):
        #print(i)
        theResult = board.step(newWhiteFrame, newBlackFrame)
        sumResults.append(theResult)
    #print("move is")
    print("----------result----------")
    timeSeriePrint(sumResults)
    print("--------------------------")
    # print(len(sumResults))
    # print("-------First State is-------")
    # print("white")
    # print(board.curStateWhite)
    # print("black")
    # print(board.curStateBlack)

In [263]:
def testCase2():
    print("test case 2")
    windowsSize = 60

    initWhiteFrame = createEmptyBoard()
    initWhiteFrame[4,4] = PID_KING
    initBlackFrame = createEmptyBoard()
    initBlackFrame[5,5] = PID_KING

    whiteFrames = duplicateFrame(initWhiteFrame, windowsSize)
    blackFrames = duplicateFrame(initBlackFrame, windowsSize)

    board = BoardData(whiteFrames, blackFrames, windowsSize, 75)
    #printAddr(board)

    newWhiteFrame = createEmptyBoard()
    newBlackFrame = createEmptyBoard() ### black king is captured
    newBlackFrame[4,4] = PID_KING
    
    
    sumResults = []
    for i in range(46):
        #print(i)
        theResult = board.step(newWhiteFrame, newBlackFrame)
        sumResults.append(theResult)
    #print("move is")
    print("----------result----------")
    timeSeriePrint(sumResults)
    print("--------------------------")
    # print(len(sumResults))
    # print("-------First State is-------")
    # print("white")
    # print(board.curStateWhite)
    # print("black")
    # print(board.curStateBlack)

In [264]:
def testCase3():
    print("test case 3")
    windowsSize = 60

    initWhiteFrame = createEmptyBoard()
    initWhiteFrame[4,4] = PID_KING
    initWhiteFrame[0,2] = PID_PAWN
    initBlackFrame = createEmptyBoard()
    initBlackFrame[5,5] = PID_KING

    whiteFrames = duplicateFrame(initWhiteFrame, windowsSize)
    blackFrames = duplicateFrame(initBlackFrame, windowsSize)

    board = BoardData(whiteFrames, blackFrames, windowsSize, 75)
    #printAddr(board)

    newWhiteFrame = createEmptyBoard()
    newWhiteFrame[0,3] = PID_PAWN
    newWhiteFrame[4,3] = PID_KING
    newBlackFrame = createEmptyBoard() ### black king is captured
    newBlackFrame[5,5] = PID_KING
    
    
    sumResults = []
    for i in range(46):
        #print(i)
        theResult = board.step(newWhiteFrame, newBlackFrame)
        sumResults.append(theResult)
    #print("move is")
    print("----------result----------")
    timeSeriePrint(sumResults)
    print("--------------------------")
    # print(len(sumResults))
    # print("-------First State is-------")
    # print("white")
    # print(board.curStateWhite)
    # print("black")
    # print(board.curStateBlack)

In [265]:
def testCase4():
    print("test case 4")
    windowsSize = 60

    initWhiteFrame = createEmptyBoard()
    initWhiteFrame[4,4] = PID_KING
    initWhiteFrame[0,2] = PID_PAWN
    initBlackFrame = createEmptyBoard()
    initBlackFrame[5,5] = PID_KING
    initBlackFrame[6,6] = PID_BISHOP

    whiteFrames = duplicateFrame(initWhiteFrame, windowsSize)
    blackFrames = duplicateFrame(initBlackFrame, windowsSize)

    board = BoardData(whiteFrames, blackFrames, windowsSize, 75)
    #printAddr(board)

    newWhiteFrame = createEmptyBoard()
    newWhiteFrame[4,4] = PID_PAWN
    newWhiteFrame[0,2] = PID_KING
    newBlackFrame = createEmptyBoard() ### black king is captured
    newBlackFrame [4,4] = PID_KING
    initBlackFrame[7,6] = PID_BISHOP
    
    
    sumResults = []
    for i in range(46):
        #print(i)
        theResult = board.step(newWhiteFrame, newBlackFrame)
        sumResults.append(theResult)
    #print("move is")
    print("----------result----------")
    timeSeriePrint(sumResults)
    print("--------------------------")
    # print(len(sumResults))
    # print("-------First State is-------")
    # print("white")
    # print(board.curStateWhite)
    # print("black")
    # print(board.curStateBlack)

In [266]:
def testCase5():
    print("test case 5")
    windowsSize = 60

    initWhiteFrame = createEmptyBoard()
    initWhiteFrame[4,4] = PID_KING
    initBlackFrame = createEmptyBoard()
    initBlackFrame[5,5] = PID_KING

    whiteFrames = duplicateFrame(initWhiteFrame, windowsSize)
    blackFrames = duplicateFrame(initBlackFrame, windowsSize)

    board = BoardData(whiteFrames, blackFrames, windowsSize, 75)
    #printAddr(board)

    newWhiteFrame = createEmptyBoard()
    newWhiteFrame[3,3] = PID_KING
    newBlackFrame = createEmptyBoard() ### black king is captured
    newBlackFrame[6,6] = PID_KING
    
    
    sumResults = []
    for i in range(46):
        #print(i)
        theResult = board.step(newWhiteFrame, newBlackFrame)
        sumResults.append(theResult)
    #print("move is")
    print("----------result----------")
    timeSeriePrint(sumResults)
    print("--------------------------")
    # print(len(sumResults))
    # print("-------First State is-------")
    # print("white")
    # print(board.curStateWhite)
    # print("black")
    # print(board.curStateBlack)

In [267]:
testCase0() # white move
testCase1() # white captures black
testCase2() # black captures white
testCase3() # 2 white move (invalid)
testCase4() # 2 black move (invalid)
testCase5() # 1 move from each side

test case 0
----------result----------
None   :   44
((((4, 4), (3, 3)), 4), (((8, 8), (8, 8)), 6))   :   1
None    :    1
--------------------------
test case 1
----------result----------
None   :   44
((((4, 4), (5, 5)), 4), (((5, 5), (8, 8)), 4))   :   1
None    :    1
--------------------------
test case 2
----------result----------
None   :   44
((((4, 4), (8, 8)), 4), (((5, 5), (4, 4)), 4))   :   1
None    :    1
--------------------------
test case 3
----------result----------
None    :    46
--------------------------
test case 4
----------result----------
None    :    46
--------------------------
test case 5
----------result----------
None    :    46
--------------------------
